# Lesson 1 | From membrane potential to a minimal computational neuron

Welcome to the first FPGA FlyBrain lesson.

We are not starting from computer engineering. We are starting from neurophysiology you have already seen: membrane potential changes, synaptic inputs affect a neuron, and under certain conditions the neuron produces an action potential. Our job is to translate those familiar ideas step by step into a **mathematical model → program → digital circuit → neural network running on real hardware**.

Today we will not learn hardware yet. We ask only one question:

> **If we want to study computation in a large neural network, what is the minimum neuronal behavior we need to keep?**

The primary new idea in this lesson is simple: **a scientific model is a purposeful simplification.**

## 1. What are you looking at? A Jupyter Notebook

You are reading a **Jupyter Notebook**. It lets us place two kinds of material on the same page:

- **Markdown cells**: textbook-like explanations, equations, questions, and conclusions;
- **Code cells**: real Python code that can be executed immediately.

So this is not “read a book, then go somewhere else to program.” The page itself is an **executable textbook**.

If you have never used Jupyter, that is fine. For now, all you need to know is that you can select a code cell, run it, and see its output. You do not need to learn all of Python before beginning.

## 2. Where are we eventually going? What is an FPGA?

Meet the destination, but do not try to master it today.

A **Field-Programmable Gate Array (FPGA)** is a chip whose internal digital circuitry can be configured after manufacturing according to our hardware design.

A **Central Processing Unit (CPU)** in a normal computer usually executes a sequence of instructions. An FPGA is different: we can arrange part of the computation directly as our own parallel digital circuit.

Eventually this project will turn neuron state, synaptic events, and network propagation into hardware structures inside an FPGA.

**Now put that idea on the shelf.** Lesson 1 is about the neuron model. The fact that our destination is an FPGA is not a reason to bury you in digital-circuit vocabulary on day one.

## 3. Start from neurophysiology you already know

You probably already know that:

1. there is an electrical potential difference across the neuronal membrane;
2. synaptic input can push the membrane state toward or away from firing;
3. under suitable conditions a neuron produces an action potential;
4. after an action potential, the neuron's state changes and refractory behavior appears.

A real neuron is much more complicated: ion channels, dendrites, neurotransmitters, adaptation, synaptic dynamics, cell-type differences, and much more all matter.

But our engineering question is:

> If we preserve every biological detail immediately, can we still see clearly how connectivity produces computation?

That is why we need a model.

## 4. What is a model?

A **model** does not claim that reality contains only the features in the model. It says:

> For the question we are asking now, we will keep some factors and temporarily ignore others.

A subway map is a useful example. It does not draw every building, but it preserves the information that matters for navigation: which stations connect to which.

Our first neuron model does the same thing. It is not trying to reconstruct full cellular biophysics. It gives us a simple computational unit that still has changing state and spike events.

## 5. What is LIF? Read the name literally

We begin with a classic simplified neuron model:

**Leaky Integrate-and-Fire (LIF)**.

The three words already describe the model:

- **Leaky**: without enough new input, the influence of previously accumulated membrane state gradually decays;
- **Integrate**: new input accumulates into the current state;
- **Fire**: once the state reaches a threshold, the model emits a spike event.

Here a **spike** means “the neuron fired at this moment.” We are not simulating the full voltage waveform of a biological action potential; we keep only the occurrence of the event.

That is our first deliberate simplification.

## 6. Express it with a discrete-time rule

We divide time into steps `t = 0, 1, 2, ...`. This is a **discrete-time** model.

At each step we first compute:

`V[t+1] = alpha × V[t] + I[t]`

What does each symbol mean?

- `V[t]`: membrane-state value stored at the beginning of step `t`;
- `I[t]`: input received during that step;
- `alpha`: the fraction of the previous state that remains. With `alpha < 1`, old state decays;
- `V[t+1]`: the candidate next state after decay and new input.

Then we ask:

`V[t+1] >= threshold ?`

If the threshold is reached, we emit a spike and reset the membrane state according to a reset rule.

We intentionally omit an explicit refractory-period state in Lesson 1 so that the smallest model remains visible. Later we will ask exactly which rules must be frozen before hardware implementation.

## 7. Translate the rule into minimal Python

The code below does only three things:

1. stores the current state `v`;
2. computes `candidate_v = alpha * v + current`;
3. checks the threshold and resets if a spike occurs.

Do not worry about every Python syntax detail yet. First check whether the code maps cleanly onto the model above.

In [ ]:
def run_lif(inputs, alpha=0.9, threshold=1.0, reset=0.0):
    v = 0.0
    trace = []

    for t, current in enumerate(inputs):
        candidate_v = alpha * v + current
        spike = candidate_v >= threshold
        stored_v = reset if spike else candidate_v

        trace.append({
            't': t,
            'input': current,
            'candidate_v': candidate_v,
            'stored_v': stored_v,
            'spike': spike,
        })

        v = stored_v

    return trace

inputs = [0.22] * 15
trace = run_lif(inputs)

for row in trace:
    print(row)

## 8. Read the result before drawing a plot

For each row, find five things:

- `t`: the current time step;
- `input`: the input during that step;
- `candidate_v`: the membrane candidate before threshold/reset;
- `spike`: whether firing occurred;
- `stored_v`: the state actually preserved for the next step.

Notice that `candidate_v` and `stored_v` may differ when a spike causes reset.

Later this becomes an important hardware idea: the computed **next state** and the state that is finally stored can be different concepts. For now, simply notice it.

## 9. Plot the membrane trajectory

Tables are precise; plots are often better for intuition. We will use the common Python plotting library `matplotlib`. You do not need to learn the plotting library itself in this lesson.

In [ ]:
import matplotlib.pyplot as plt

times = [row['t'] for row in trace]
candidate_v = [row['candidate_v'] for row in trace]
spike_times = [row['t'] for row in trace if row['spike']]
spike_values = [row['candidate_v'] for row in trace if row['spike']]

plt.figure(figsize=(9, 4))
plt.plot(times, candidate_v, marker='o', label='membrane candidate V')
plt.axhline(1.0, linestyle='--', label='threshold')
plt.scatter(spike_times, spike_values, marker='x', s=80, label='spike event')
plt.xlabel('time step')
plt.ylabel('model membrane state')
plt.title('A minimal LIF neuron')
plt.legend()
plt.show()

## 10. Observe: what should you actually notice?

Do not stop at “the plot appeared.” Answer:

1. With constant input, why does the membrane state not simply increase by `0.22` every step?
2. What effect does `alpha = 0.9` create?
3. Why is a spike represented as an event rather than a full action-potential waveform?
4. Why does the next accumulation begin from a lower state after firing?

If you cannot explain these, stay here before modifying the code.

## 11. Try It: predict first, then run

Change only one thing at a time:

- change `alpha` from `0.9` to `0.5`;
- or change `threshold` from `1.0` to `1.5`;
- or change each input from `0.22` to `0.35`.

**Before running, write down your prediction: will the spike be earlier, later, or disappear? Why?**

Engineering learning is not “change a parameter and see what happens.” It is “predict from the model, then use the experiment to test your understanding.”

## 12. AI Task

You may ask AI to:

- format the trace more clearly;
- compare several `alpha` values on a plot;
- add a refractory-period experimental version;
- draft tests for no-input decay, threshold crossing, and reset.

Give it one constraint:

> Do not silently change the current LIF update order, threshold rule, or reset rule. If a change seems useful, explain why and treat it as a proposed specification change.

AI may help us write the implementation, but it must not secretly define the model.

## 13. Human Check

Without AI, you should be able to explain:

- what Leaky / Integrate / Fire each means;
- whether `V` is a complete copy of biological membrane voltage, and why not;
- what `alpha` means in this simplified model;
- why a spike is an event here rather than an action-potential waveform;
- whether threshold and reset are automatically given by nature or must be explicitly defined model rules;
- which biological details this model intentionally ignores.

## 14. Engineering Handoff

The code in this Notebook is still a teaching prototype. After the model semantics are confirmed, the formal implementation will live in:

`python/reference/lif_float.py`

The Notebook should later import the formal module instead of maintaining a slightly different second implementation.

## 15. Project Trace

This section is for project maintainers and your future self; it is not vocabulary to memorize.

- Lesson ID: `LSN-001`
- Engineering slice: `RMD-001`
- Product requirement/design: `FR1 / DP1`
- Initial tests: `T-001 ~ T-004`

These IDs keep lessons, requirements, code, and tests traceable over time.

## 16. Exit Ticket

Before continuing, you should be able to:

1. explain what a scientific model is in your own words;
2. expand and explain **Leaky Integrate-and-Fire (LIF)**;
3. explain `V`, `I`, `alpha`, threshold, and reset;
4. manually step through a short input sequence and predict membrane state/spikes;
5. explain what the model keeps and what it deliberately leaves out.

If that is comfortable, the next lesson asks:

> When `0.9`, `0.22`, and `1.0` enter real digital hardware, how are those numbers actually stored?